# Task 7: ResNet-50 Style Residual Bottleneck Block and Grouped Convolutions

**Objective:** Build high-performance bottleneck projection networks and multi-channel grouped convolutions, running PyTorch Profiler to analyze computational execution costs.

In [ ]:
import torch
import torch.nn as nn
import torch.autograd.profiler as profiler

class ResNetBottleneckBlock(nn.Module):
    def __init__(self, in_channels, bottleneck_channels, out_channels, stride=1, groups=1):
        super().__init__()
        
        # 1. 1x1 Convolution: dimension reduction
        self.conv1 = nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)
        
        # 2. 3x3 Convolution: spatial processing (supports grouped convolutions)
        self.conv2 = nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, 
                               stride=stride, padding=1, groups=groups, bias=False)
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)
        
        # 3. 1x1 Convolution: dimension expansion
        self.conv3 = nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        
        self.relu = nn.ReLU(inplace=True)
        
        # Skip projection layer if input/output dimensions mismatch
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
            
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        
        out += residual
        out = self.relu(out)
        return out

In [ ]:
# Instanciate blocks with different settings
in_c, bot_c, out_c = 256, 64, 256
input_tensor = torch.randn(8, in_c, 56, 56) # standard size in ResNet-50

standard_block = ResNetBottleneckBlock(in_c, bot_c, out_c, groups=1)
grouped_block = ResNetBottleneckBlock(in_c, bot_c, out_c, groups=32) # Grouped Conv (cardinality=32)
depthwise_block = ResNetBottleneckBlock(in_c, bot_c, out_c, groups=bot_c) # Depthwise Conv in bottleneck

# Count parameters
p_std = sum(p.numel() for p in standard_block.parameters())
p_grp = sum(p.numel() for p in grouped_block.parameters())
p_dth = sum(p.numel() for p in depthwise_block.parameters())

print(f"Standard Bottleneck Parameters: {p_std:,}")
print(f"Grouped Bottleneck Parameters:  {p_grp:,}")
print(f"Depthwise Bottleneck Parameters:{p_dth:,}")

# Profile execution using PyTorch Profiler
with profiler.profile(record_shapes=True) as prof_std:
    standard_block(input_tensor)

with profiler.profile(record_shapes=True) as prof_grp:
    grouped_block(input_tensor)

with profiler.profile(record_shapes=True) as prof_dth:
    depthwise_block(input_tensor)

print("\n--- Profiler Execution Summary (Total CPU Time) ---")
print(f"Standard Block CPU Time:  {prof_std.self_cpu_time_total / 1000.0:.2f} ms")
print(f"Grouped Block CPU Time:   {prof_grp.self_cpu_time_total / 1000.0:.2f} ms")
print(f"Depthwise Block CPU Time: {prof_dth.self_cpu_time_total / 1000.0:.2f} ms")